# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alizawwaris974/Assignment-1---Flyrank-ML/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import duckdb, getpass
import pandas as pd, numpy as np

HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ').strip()
conn = duckdb.connect()
conn.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

# Same Feb (prior) vs March (current) momentum build as w04_baseline_score
momentum_query = f"""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_prev
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_last,
               SUM(gsc_clicks) AS clicks_mar,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_mar
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_impressions IS NOT NULL
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT mar.*, feb.imp_prev
    FROM mar LEFT JOIN feb
      ON mar.client_hash_id = feb.client_hash_id AND mar.content_hash_id = feb.content_hash_id
    WHERE mar.imp_last IS NOT NULL AND feb.imp_prev >= 100
"""
signal_df = conn.execute(momentum_query).df()

content_query = f"SELECT content_hash_id, content_created_date, content_updated_date FROM read_parquet('{REL}/dim_content.parquet')"
content_df = conn.execute(content_query).df()
signal_df = signal_df.merge(content_df, on="content_hash_id", how="left")

signal_df["is_declining"] = (signal_df["imp_last"] < 0.8 * signal_df["imp_prev"]).astype(int)
signal_df["ctr_mar"] = signal_df["clicks_mar"] / signal_df["imp_last"]
signal_df["content_age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(signal_df["content_created_date"])).dt.days

print(signal_df.shape, "| decline rate:", signal_df["is_declining"].mean().round(3))

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(76837, 11) | decline rate: 0.182


In [2]:
for col in ["imp_prev", "imp_last", "clicks_mar", "content_age_days"]:
    print(f"\n{col}:")
    print(signal_df[col].describe())
    print("median:", signal_df[col].median(), "| mean/median ratio:", round(signal_df[col].mean() / max(signal_df[col].median(), 1), 2))


imp_prev:
count     76837.000000
mean       2294.238830
std        5453.148392
min         100.000000
25%         278.000000
50%         742.000000
75%        2235.000000
max      203401.000000
Name: imp_prev, dtype: float64
median: 742.0 | mean/median ratio: 3.09

imp_last:
count     76837.000000
mean       3317.422271
std        7804.147891
min           0.000000
25%         374.000000
50%        1067.000000
75%        3274.000000
max      617124.000000
Name: imp_last, dtype: float64
median: 1067.0 | mean/median ratio: 3.11

clicks_mar:
count    76837.000000
mean         9.445736
std         39.336162
min          0.000000
25%          0.000000
50%          2.000000
75%          7.000000
max       5668.000000
Name: clicks_mar, dtype: float64
median: 2.0 | mean/median ratio: 4.72

content_age_days:
count    76837.000000
mean       217.206021
std        116.517171
min         33.000000
25%        110.000000
50%        215.000000
75%        287.000000
max        494.000000
Name: conten

In [3]:
print(signal_df["content_created_date"].value_counts().head(5))

content_created_date
2025-10-16    2876
2025-09-19    2076
2025-03-21    2019
2026-01-16    1950
2025-02-12    1828
Name: count, dtype: int64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [4]:
def bucket_check(df, col, bins, labels, label_col="is_declining"):
    b = pd.cut(df[col], bins=bins, labels=labels)
    out = df.groupby(b, observed=True).agg(n=(label_col, "size"), decline_rate=(label_col, "mean"))
    print(out)
    print("base rate:", df[label_col].mean().round(3))
    return out

print("=== Test 1: position vs decline ===")
bucket_check(signal_df, "avg_position_mar", [0, 3, 10, 20, 50, 10000], ["top_3", "page_1", "striking", "page_3_5", "deep"])

print("\n=== Test 2: staleness (content_updated_date) vs decline ===")
signal_df["days_since_update"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(signal_df["content_updated_date"])).dt.days
valid_update = signal_df[signal_df["days_since_update"] >= 0]
print(f"Valid (non-future-dated) rows: {len(valid_update)} of {len(signal_df)}")
bucket_check(valid_update, "days_since_update", [-1, 30, 90, 180, 10000], ["0-30d", "31-90d", "91-180d", "180d+"])

print("\n=== Test 3: content age vs decline ===")
bucket_check(signal_df, "content_age_days", [0, 90, 180, 365, 10000], ["0-90d", "91-180d", "181-365d", "365d+"])

=== Test 1: position vs decline ===
                      n  decline_rate
avg_position_mar                     
top_3              5872      0.118358
page_1            36289      0.178704
striking          17241      0.211299
page_3_5          15695      0.171010
deep               1590      0.208805
base rate: 0.182

=== Test 2: staleness (content_updated_date) vs decline ===
Valid (non-future-dated) rows: 16799 of 76837
                       n  decline_rate
days_since_update                     
0-30d                 77      0.506494
31-90d             16637      0.228407
91-180d               58      0.034483
180d+                 27      0.666667
base rate: 0.23

=== Test 3: content age vs decline ===
                      n  decline_rate
content_age_days                     
0-90d             15472      0.130752
91-180d           14882      0.216436
181-365d          33902      0.186980
365d+             12581      0.191241
base rate: 0.182


,n,decline_rate
content_age_days,,
0-90d,15472,0.130752
91-180d,14882,0.216436
181-365d,33902,0.186980
365d+,12581,0.191241


Test 1 (position): MIXED — same reasoning as w04_baseline_score: directionally rising, not strictly monotonic, all buckets well-powered.

Test 2 (staleness via content_updated_date): FALSE — same finding as before: mostly future-dated, and the valid subset is 97% a single bulk-update date. Reuse that exact reasoning here since it's the same underlying column.

Test 3 (content age): No bulk artifact contamination issue

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [5]:
signal_df["position_bucket"] = pd.cut(signal_df["avg_position_mar"], bins=[0, 3, 10, 20, 50, 10000],
                                        labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
ctr_by_tier = signal_df.groupby("position_bucket", observed=True).agg(
    n=("ctr_mar", "size"),
    mean_ctr=("ctr_mar", "mean")
)
print(ctr_by_tier)

                     n  mean_ctr
position_bucket                 
top_3             5872  0.003385
page_1           36289  0.002977
striking         17241  0.002271
page_3_5         15695  0.001300
deep              1590  0.000425


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

position and CTR-vs-position are trustworthy signals a content team can act on directly. content_updated_date-based staleness is not reliable
in this dataset due to a bulk-update artifact concentrated on one date — a refresh flag built on
it would mostly be detecting who was touched in that migration, not genuine neglect.

## Self-check

Before you submit, confirm each line honestly:

- [T] Every section above is filled — markdown thinking AND the code that backs it
- [T] The notebook runs top to bottom with no errors (Runtime → Run all)
- [T] No client names, URLs, or private queries anywhere
- [T] My claims use careful words: observed, measured, directional, decision-support
- [T] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.